# P1-E09.2 — CAUEEG Exact-Age-Matched and Age-Stratified Sensitivity Lock

**Paper 1, Experiment 09.2 — targeted age-confounding sensitivity analysis**

## Goal

Determine how much of the locked EEG benchmark remains after directly balancing age distributions across diagnostic classes. This notebook responds to the material age-confounding signal identified in P1-E09.1.

It uses only the combined official training + validation development cohort. At every rounded single-year age shared by all classes in an endpoint, it samples the same number of recordings from every class. Twenty deterministic matched samples quantify sampling variability. Within every sample, the exact P1-E08.1 representation, estimator and hyperparameters are evaluated with five-fold cross-validation. A fixed age-only logistic model is evaluated on the same matched samples.

The official test split is not used, inspected for outcomes or evaluated.

Run all cells in order. Continue only if the final status is:

> **P1-E09.2 PASSED — EXACT-AGE-MATCHED SENSITIVITY LOCKED**

## Context & Methods

### Prespecified rules

- Endpoints: Normal–MCI–AD, Normal vs AD, and MCI vs AD.
- Development data only: official training and validation recordings.
- Matching variable: age rounded to the nearest completed year.
- Common support: ages represented in every class of the endpoint.
- Within each shared age: retain the minimum class count from every class.
- Matching repetitions: 20 deterministic random samples.
- Evaluation: five-fold stratified cross-validation per matched sample.
- Models and feature subsets: exactly those locked by P1-E08.1.
- No new model selection, hyperparameter tuning or preprocessing changes.
- Age balance target: maximum absolute pairwise standardized mean difference below 0.10.
- Sex remains unavailable and cannot be claimed as controlled.
- Analysis remains recording-level because participant identity is unverified.

## Setup

### 1. Paths and dependencies

In [ ]:
from pathlib import Path
import hashlib, json, platform, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef
import joblib

PAPER_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
E08_ROOT=PAPER_ROOT/'p1_e08_1_leakage_free_nonspiking_benchmark'
E09_ROOT=PAPER_ROOT/'p1_e09_1_benchmark_stability_confound_error_audit'
OUTPUT_ROOT=PAPER_ROOT/'p1_e09_2_exact_age_matched_sensitivity'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'
for path in (OUTPUT_ROOT,TABLES,FIGURES): path.mkdir(parents=True,exist_ok=True)

E09_RUN_RECORD=E09_ROOT/'run_record.json'
E09_STABILITY=E09_ROOT/'tables'/'03_stability_and_sensitivity_summary.csv'
E09_OOF=E09_ROOT/'tables'/'04_repeated_out_of_fold_predictions.csv'
FEATURE_MATRIX=E08_ROOT/'features'/'02_LOCKED_RECORDING_LEVEL_NONSPIKING_FEATURE_MATRIX.npz'
FEATURE_METADATA=E08_ROOT/'tables'/'05_LOCKED_RECORDING_FEATURE_METADATA_WITH_LABELS.csv'

TASKS={'three_class':['Normal','MCI','AD'],'normal_vs_ad':['Normal','AD'],'mci_vs_ad':['MCI','AD']}
MATCH_REPEATS=20; FOLDS=5; SEED=20260828; MAX_ABS_SMD=0.10; MIN_PER_CLASS=30

plt.rcParams.update({'figure.dpi':120,'savefig.dpi':600,'font.size':9,
                     'axes.titlesize':10,'axes.labelsize':9,'pdf.fonttype':42,'ps.fonttype':42})
sns.set_theme(style='whitegrid',context='paper')

def sha256_file(path,block_size=4*1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def save_table(frame,name):
    frame.to_csv(TABLES/f'{name}.csv',index=False)
    try: frame.to_excel(TABLES/f'{name}.xlsx',index=False)
    except Exception as exc: warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')
def savefig(fig,name,width=8,height=4):
    fig.set_size_inches(width,height); fig.tight_layout()
    fig.savefig(FIGURES/f'{name}.png',dpi=600,bbox_inches='tight',facecolor='white')
    fig.savefig(FIGURES/f'{name}.pdf',bbox_inches='tight',facecolor='white')
def metric_set(y_true,y_pred):
    return {'accuracy':accuracy_score(y_true,y_pred),'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),
            'macro_f1':f1_score(y_true,y_pred,average='macro'),'mcc':matthews_corrcoef(y_true,y_pred)}

print('Experiment: P1-E09.2')
print('Output:',OUTPUT_ROOT)

## Data

### 2. Authenticate P1-E09.1 and the locked P1-E08.1 objects

In [ ]:
model_paths={task:E08_ROOT/'models'/f'{task}_LOCKED_NONSPIKING_BENCHMARK.joblib' for task in TASKS}
required=[E09_RUN_RECORD,E09_STABILITY,E09_OOF,FEATURE_MATRIX,FEATURE_METADATA,*model_paths.values()]
missing=[str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError('Required locked inputs are missing:\n'+'\n'.join(missing))

e09_run=json.loads(E09_RUN_RECORD.read_text(encoding='utf-8'))
stability=pd.read_csv(E09_STABILITY)
oof=pd.read_csv(E09_OOF,dtype={'recording_id':str})
metadata=pd.read_csv(FEATURE_METADATA,dtype={'recording_id':str,'serial':str})
with np.load(FEATURE_MATRIX,allow_pickle=False) as payload:
    X=np.asarray(payload['X'],dtype=float); feature_ids=payload['recording_id'].astype(str)
for column in ('recording_id','serial'):
    if column in metadata: metadata[column]=metadata[column].astype(str).str.strip().str.zfill(5)
oof.recording_id=oof.recording_id.astype(str).str.strip().str.zfill(5)
metadata['age']=pd.to_numeric(metadata.age,errors='coerce')
if metadata.recording_id.tolist()!=feature_ids.tolist(): raise ValueError('Feature order mismatch')
artifacts={task:joblib.load(path) for task,path in model_paths.items()}

input_checks=pd.DataFrame([
 {'check':'P1-E09.1 passed','pass':str(e09_run.get('status','')).startswith('P1-E09.1 PASSED'),'detail':e09_run.get('status','')},
 {'check':'stability summary authenticated','pass':sha256_file(E09_STABILITY)==e09_run.get('stability_summary_sha256'),'detail':sha256_file(E09_STABILITY)},
 {'check':'feature matrix authenticated','pass':sha256_file(FEATURE_MATRIX)==e09_run.get('source_feature_matrix_sha256'),'detail':sha256_file(FEATURE_MATRIX)},
 {'check':'914 finite recording features','pass':X.shape[0]==914 and np.isfinite(X).all(),'detail':X.shape},
 {'check':'age complete','pass':metadata.age.notna().all(),'detail':int(metadata.age.isna().sum())},
 {'check':'three locked model artifacts loaded','pass':set(artifacts)==set(TASKS),'detail':list(artifacts)},
 {'check':'P1-E09.1 OOF table structurally complete','pass':len(oof)==73310 and not oof.duplicated(['task','scenario','repeat','recording_id']).any(),'detail':len(oof)},
 {'check':'analysis remains recording-level','pass':metadata.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':metadata.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~metadata.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':metadata.participant_identity_verified.value_counts(dropna=False).to_dict()},
])
save_table(input_checks,'01_input_authentication_checks'); display(input_checks)
if not input_checks['pass'].all(): raise AssertionError('STOP — INPUT AUTHENTICATION FAILED')
print('P1-E09.1 AND P1-E08.1 LOCKS AUTHENTICATED')

## Results

### 3. Construct repeated exact-age-matched development cohorts

In [ ]:
def exact_age_match(frame,classes,seed):
    work=frame.loc[frame.experimental_label.isin(classes) &
                   frame.official_split.isin(['train','validation'])].copy()
    work['age_year']=np.rint(work.age).astype(int)
    counts=work.groupby(['age_year','experimental_label']).size().unstack(fill_value=0).reindex(columns=classes,fill_value=0)
    shared=counts.index[counts.gt(0).all(axis=1)]
    rng=np.random.default_rng(seed); selected_indices=[]
    for age_year in shared:
        n=int(counts.loc[age_year].min())
        for label in classes:
            candidates=work.index[(work.age_year==age_year)&(work.experimental_label==label)].to_numpy()
            selected_indices.extend(rng.choice(candidates,size=n,replace=False).tolist())
    return work.loc[sorted(selected_indices)].copy(),shared.tolist()

def maximum_pairwise_smd(frame,classes):
    values=[]
    for i,left in enumerate(classes):
        for right in classes[i+1:]:
            a=frame.loc[frame.experimental_label.eq(left),'age'].to_numpy(float)
            b=frame.loc[frame.experimental_label.eq(right),'age'].to_numpy(float)
            pooled=np.sqrt((np.var(a,ddof=1)+np.var(b,ddof=1))/2)+1e-12
            values.append(abs(np.mean(a)-np.mean(b))/pooled)
    return float(max(values)) if values else 0.0

match_rows=[]; balance_rows=[]
for task,classes in TASKS.items():
    for repeat in range(MATCH_REPEATS):
        matched,shared_ages=exact_age_match(metadata,classes,SEED+1000*list(TASKS).index(task)+repeat)
        counts=matched.experimental_label.value_counts().reindex(classes,fill_value=0)
        balance_rows.append({'task':task,'match_repeat':repeat,'recordings':len(matched),
            'per_class':int(counts.min()),'shared_age_years':len(shared_ages),
            'maximum_absolute_age_smd':maximum_pairwise_smd(matched,classes),
            'minimum_age':float(matched.age.min()),'maximum_age':float(matched.age.max())})
        for row in matched[['recording_id','experimental_label','age']].itertuples(index=False):
            match_rows.append({'task':task,'match_repeat':repeat,'recording_id':row.recording_id,
                               'experimental_label':row.experimental_label,'age':row.age})

match_manifest=pd.DataFrame(match_rows); balance=pd.DataFrame(balance_rows)
save_table(match_manifest,'02_repeated_exact_age_match_manifest')
save_table(balance,'03_age_balance_diagnostics')
display(balance.groupby('task').agg(min_per_class=('per_class','min'),max_per_class=('per_class','max'),
    mean_recordings=('recordings','mean'),max_abs_smd=('maximum_absolute_age_smd','max'),
    shared_age_years=('shared_age_years','min')))

### 4. Evaluate locked EEG and age-only models within every matched cohort

In [ ]:
result_rows=[]; prediction_rows=[]
id_to_index=pd.Series(np.arange(len(metadata)),index=metadata.recording_id).to_dict()
for task,classes in TASKS.items():
    artifact=artifacts[task]; columns=np.asarray(artifact['feature_indices'],dtype=int)
    for repeat in range(MATCH_REPEATS):
        ids=match_manifest.loc[(match_manifest.task==task)&(match_manifest.match_repeat==repeat),'recording_id']
        indices=np.asarray([id_to_index[value] for value in ids],dtype=int)
        y=metadata.experimental_label.to_numpy()[indices]
        for scenario in ['locked_eeg','age_only']:
            true_all=[]; pred_all=[]
            splitter=StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=SEED+repeat)
            for fold,(train_local,valid_local) in enumerate(splitter.split(indices,y),1):
                train_index=indices[train_local]; valid_index=indices[valid_local]
                if scenario=='locked_eeg':
                    train_matrix=X[train_index][:,columns]; valid_matrix=X[valid_index][:,columns]
                    estimator=clone(artifact['estimator'])
                else:
                    train_matrix=metadata.age.to_numpy(float)[train_index,None]
                    valid_matrix=metadata.age.to_numpy(float)[valid_index,None]
                    estimator=Pipeline([('scale',StandardScaler()),('model',LogisticRegression(
                        C=1.0,class_weight='balanced',max_iter=4000,random_state=SEED))])
                y_train=metadata.experimental_label.to_numpy()[train_index]
                y_valid=metadata.experimental_label.to_numpy()[valid_index]
                estimator.fit(train_matrix,y_train); prediction=estimator.predict(valid_matrix)
                true_all.extend(y_valid); pred_all.extend(prediction)
                for global_index,true,pred in zip(valid_index,y_valid,prediction):
                    prediction_rows.append({'task':task,'match_repeat':repeat,'scenario':scenario,
                        'fold':fold,'recording_id':metadata.recording_id.iloc[global_index],
                        'true_label':true,'predicted_label':pred,'correct':bool(true==pred)})
            result_rows.append({'task':task,'match_repeat':repeat,'scenario':scenario,
                                'n_recordings':len(indices),**metric_set(true_all,pred_all)})
    print(task,'matched evaluations complete')

matched_results=pd.DataFrame(result_rows); matched_predictions=pd.DataFrame(prediction_rows)
matched_summary=(matched_results.groupby(['task','scenario']).agg(
    matching_repeats=('match_repeat','nunique'),mean_n=('n_recordings','mean'),
    mean_balanced_accuracy=('balanced_accuracy','mean'),sd_balanced_accuracy=('balanced_accuracy','std'),
    p025_balanced_accuracy=('balanced_accuracy',lambda x:x.quantile(.025)),
    p975_balanced_accuracy=('balanced_accuracy',lambda x:x.quantile(.975)),
    mean_macro_f1=('macro_f1','mean'),mean_mcc=('mcc','mean')).reset_index())

original=(stability.loc[stability.scenario.eq('locked_eeg'),['task','mean_balanced_accuracy']]
          .rename(columns={'mean_balanced_accuracy':'unmatched_locked_eeg_ba'}))
comparison=matched_summary.merge(original,on='task',how='left',validate='many_to_one')
comparison['matched_minus_unmatched_ba']=comparison.mean_balanced_accuracy-comparison.unmatched_locked_eeg_ba
save_table(matched_results,'04_exact_age_matched_repeat_metrics')
save_table(matched_predictions,'05_exact_age_matched_oof_predictions')
save_table(matched_summary,'06_exact_age_matched_summary')
save_table(comparison,'07_matched_vs_unmatched_comparison')
display(comparison)

### 5. Age-stratified performance using P1-E09.1 locked out-of-fold predictions

In [ ]:
locked_oof=oof.loc[oof.scenario.eq('locked_eeg')].copy()
locked_oof=locked_oof.merge(metadata[['recording_id','age']],on='recording_id',how='left',validate='many_to_one')
strata_rows=[]; boundary_rows=[]
for task,classes in TASKS.items():
    development=metadata.loc[metadata.experimental_label.isin(classes)&metadata.official_split.isin(['train','validation'])]
    boundaries=np.unique(np.quantile(development.age,[0,.25,.5,.75,1]))
    if len(boundaries)<5: boundaries=np.linspace(development.age.min(),development.age.max()+1e-9,5)
    boundary_rows.append({'task':task,'boundaries':'|'.join(f'{value:.3f}' for value in boundaries)})
    task_oof=locked_oof.loc[locked_oof.task.eq(task)].copy()
    task_oof['age_stratum']=pd.cut(task_oof.age,bins=boundaries,include_lowest=True,duplicates='drop')
    for (repeat,stratum),group in task_oof.groupby(['repeat','age_stratum'],observed=True):
        if group.true_label.nunique()<2: continue
        strata_rows.append({'task':task,'repeat':repeat,'age_stratum':str(stratum),
            'n_predictions':len(group),'n_recordings':group.recording_id.nunique(),
            **metric_set(group.true_label,group.predicted_label)})

age_stratified=pd.DataFrame(strata_rows)
age_stratified_summary=(age_stratified.groupby(['task','age_stratum']).agg(
    mean_n_recordings=('n_recordings','mean'),repeats=('repeat','nunique'),
    mean_balanced_accuracy=('balanced_accuracy','mean'),sd_balanced_accuracy=('balanced_accuracy','std'),
    mean_macro_f1=('macro_f1','mean'),mean_mcc=('mcc','mean')).reset_index())
save_table(pd.DataFrame(boundary_rows),'08_age_stratum_boundaries')
save_table(age_stratified,'09_age_stratified_repeat_metrics')
save_table(age_stratified_summary,'10_age_stratified_summary')
display(age_stratified_summary)

### 6. Publication-ready age-sensitivity figures

In [ ]:
fig,axes=plt.subplots(1,3)
for ax,(task,group) in zip(axes,matched_results.groupby('task')):
    sns.boxplot(data=group,x='scenario',y='balanced_accuracy',hue='scenario',legend=False,ax=ax)
    unmatched=float(original.loc[original.task.eq(task),'unmatched_locked_eeg_ba'].iloc[0])
    ax.axhline(unmatched,color='#D55E00',ls='--',label='Unmatched EEG')
    ax.set(title=task.replace('_',' '),xlabel='',ylabel='Balanced accuracy'); ax.legend(frameon=False,fontsize=7)
savefig(fig,'Figure_01_exact_age_matched_performance',13,4); plt.show()

fig,axes=plt.subplots(1,3)
for ax,(task,group) in zip(axes,age_stratified_summary.groupby('task')):
    ax.errorbar(np.arange(len(group)),group.mean_balanced_accuracy,yerr=group.sd_balanced_accuracy,
                marker='o',capsize=3,color='#0072B2')
    ax.set(xticks=np.arange(len(group)),xticklabels=group.age_stratum,ylim=(0,1),
           title=task.replace('_',' '),xlabel='Development age stratum',ylabel='Balanced accuracy')
    ax.tick_params(axis='x',rotation=35)
savefig(fig,'Figure_02_age_stratified_performance',14,4.5); plt.show()

first_matches=match_manifest.loc[match_manifest.match_repeat.eq(0)]
fig,axes=plt.subplots(1,3)
for ax,(task,group) in zip(axes,first_matches.groupby('task')):
    sns.histplot(data=group,x='age',hue='experimental_label',multiple='dodge',binwidth=1,ax=ax)
    ax.set(title=task.replace('_',' '),xlabel='Age',ylabel='Matched recordings')
savefig(fig,'Figure_03_exact_age_balance_example',14,4); plt.show()

## Checks

### 7. Final age-sensitivity integrity gates

In [ ]:
expected_results=len(TASKS)*MATCH_REPEATS*2
checks=pd.DataFrame([
 {'check':'P1-E09.1 source authenticated','pass':input_checks['pass'].all(),'detail':'all source gates passed'},
 {'check':'all tasks completed 20 matches','pass':balance.groupby('task').match_repeat.nunique().eq(MATCH_REPEATS).all(),'detail':balance.groupby('task').match_repeat.nunique().to_dict()},
 {'check':'minimum matched sample adequate','pass':balance.per_class.ge(MIN_PER_CLASS).all(),'detail':balance.groupby('task').per_class.min().to_dict()},
 {'check':'exact matching produced equal class counts','pass':match_manifest.groupby(['task','match_repeat','experimental_label']).size().groupby(['task','match_repeat']).nunique().eq(1).all(),'detail':'equal within every task/repeat'},
 {'check':'maximum age SMD below 0.10','pass':balance.maximum_absolute_age_smd.le(MAX_ABS_SMD).all(),'detail':balance.groupby('task').maximum_absolute_age_smd.max().to_dict()},
 {'check':'locked EEG and age-only evaluations complete','pass':len(matched_results)==expected_results,'detail':len(matched_results)},
 {'check':'one OOF prediction per matched recording/scenario','pass':not matched_predictions.duplicated(['task','match_repeat','scenario','recording_id']).any(),'detail':len(matched_predictions)},
 {'check':'age-only result available for every task','pass':set(matched_summary.loc[matched_summary.scenario.eq('age_only'),'task'])==set(TASKS),'detail':'three tasks'},
 {'check':'age-stratified analysis completed','pass':set(age_stratified_summary.task)==set(TASKS),'detail':age_stratified_summary.groupby('task').size().to_dict()},
 {'check':'official test split excluded from matching and evaluation','pass':'official_split' in metadata and not metadata.loc[metadata.official_split.eq('test'),'recording_id'].isin(match_manifest.recording_id).any(),'detail':'development cohort only'},
 {'check':'no new hyperparameter selection performed','pass':True,'detail':'cloned locked estimators'},
 {'check':'sex confounding not falsely assessed','pass':True,'detail':'sex unavailable'},
 {'check':'analysis remains recording-level','pass':metadata.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':metadata.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~metadata.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':metadata.participant_identity_verified.value_counts(dropna=False).to_dict()},
])
save_table(checks,'11_final_integrity_checks'); display(checks)
status=('P1-E09.2 PASSED — EXACT-AGE-MATCHED SENSITIVITY LOCKED'
        if checks['pass'].all() else 'STOP — P1-E09.2 CHECKS FAILED; DO NOT FINALISE CONFOUND CLAIMS')

matched_csv=TABLES/'06_exact_age_matched_summary.csv'; stratified_csv=TABLES/'10_age_stratified_summary.csv'
run_record={'experiment':'P1-E09.2 CAUEEG exact-age-matched and age-stratified sensitivity lock',
 'timestamp_utc':pd.Timestamp.utcnow().isoformat(),'status':status,
 'source_p1_e09_run_record':str(E09_RUN_RECORD),'source_stability_summary_sha256':sha256_file(E09_STABILITY),
 'matching_rule':'rounded single-year exact frequency matching across all endpoint classes',
 'matching_repeats':MATCH_REPEATS,'folds_per_match':FOLDS,'minimum_per_class':MIN_PER_CLASS,
 'matched_summary_csv':str(matched_csv),'matched_summary_sha256':sha256_file(matched_csv),
 'age_stratified_summary_csv':str(stratified_csv),'age_stratified_summary_sha256':sha256_file(stratified_csv),
 'official_test_used':False,'new_hyperparameter_selection':False,
 'sex_confound_assessment':'unavailable in released CAUEEG',
 'identity_interpretation':{'analysis_unit':'recording','participant_identity_verified':False},
 'python':sys.version,'platform':platform.platform()}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))
if not checks['pass'].all(): raise AssertionError(status)